# Refining the activity forecasts

> **Historical v1 analysis.** The modelling target in this workflow ends in September 2025; newer supplemental observations remain separate here. For the current 519-game forecast, see [Notebook 20](20_fixed_cohort_2027_forecast.ipynb). [Notebook 17](17_fixed_sources_integration.ipynb) begins the separate v2 workflow.

The earlier comparison showed uneven performance across categories and cutoffs. Recent movement and variability may help distinguish a stable activity level from a changing one, so each was added separately to the existing lag-based model. An elapsed-month feature was also tested to capture changes across the historical period. Model settings, training rows and evaluation dates were held constant.

## Comparison with the existing model

Each variant retains the original lags, rolling averages and calendar fields. Recent changes adds one- and three-month absolute movements; rolling variability adds trailing three- and twelve-month standard deviations. All future activity-derived inputs are calculated from the recursive forecast history. The table reports MAE for leads 16–27.

In [1]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
scores = pd.read_csv(ROOT / "reports/feature_experiments/scores.csv")
comparison = scores.loc[scores.window.eq("Leads 16 to 27")].pivot(index="category", columns="model", values="mae").round(2).reset_index()
comparison


model,category,Recent changes,Reference,Rolling variability,Time trend
,Action,450805.19,341764.52,465443.00,279129.06
,Indie,58655.78,54346.87,68467.77,53976.45
,Massively Multiplayer,87638.04,66380.81,87583.61,66128.70
,RPG,53440.61,58620.57,52369.98,58213.65
,Strategy,31327.06,39031.61,32735.98,39029.39


Recent changes improves Strategy by 19.7% against the original ML model and by about 2.6% against last value. Rolling variability improves RPG by 10.7% against the ML reference and about 8.2% against last value. Neither addition helps consistently across categories: both increase errors for Action and Massively Multiplayer.

## Consistency across cutoffs

Strategy benefits from recent changes at three cutoffs. RPG benefits from variability at only one cutoff, where a large reduction outweighs the other losses. These differences matter when deciding whether a lower average error is dependable.

In [2]:
stability = pd.read_csv(ROOT / "reports/feature_experiments/stability.csv")
stability.round(2)

category,model,worst_fold_mae,wins_vs_reference
Action,Recent changes,528785.19,0
Action,Reference,490602.71,0
Action,Rolling variability,599302.50,0
Action,Time trend,469176.32,2
Indie,Recent changes,62782.12,1
Indie,Reference,63895.06,0
Indie,Rolling variability,102650.03,1
Indie,Time trend,63895.06,1
Massively Multiplayer,Recent changes,139221.65,0
Massively Multiplayer,Reference,99577.28,0


The elapsed-month feature improves Action relative to the ML reference, but does not beat its last-value forecast. Changes for Indie and Massively Multiplayer are small. The evidence at this stage supported category-specific investigation rather than adopting a larger feature set throughout.

## Decision at this stage

Feature additions remained candidates at this stage. The same historical windows have informed development, so these improvements required further robustness assessment before selection. The following tuning analysis compared a bounded set of candidates with the wider model results.

Recorded evidence: [feature-experiment manifest](../reports/feature_experiments/manifest.json).